<a href="https://colab.research.google.com/github/RyuichiSaito1/inflation-reddit-usa/blob/main/notebooks/vader_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 0: Install and import required libraries

!pip install nltk pandas gspread gspread_dataframe --quiet

import os
import pandas as pd
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import auth
from google.auth import default


In [ ]:
# Step 0-1: Download VADER lexicon if not already downloaded

nltk.download("vader_lexicon")


In [ ]:
# Step 1: Set paths and time range parameters

# Input directory and file paths
BASE_DIR = "/content/drive/MyDrive/world-inflation/data/reddit/production/"
COMMENTS_FILE = os.path.join(BASE_DIR, "Frugal_comments_2012_2022.tsv")
SUBMISSIONS_FILE = os.path.join(BASE_DIR, "Frugal_submissions_2012_2022.tsv")

# Google Spreadsheet settings
SPREADSHEET_NAME = "monthly_sentiment_results"
WORKSHEET_NAME = "frugal"

# Time range (inclusive). Format: "YYYY-MM"
START_MONTH = "2012-01"
END_MONTH = "2022-12"

# Convert to concrete datetime range
start_date = pd.to_datetime(START_MONTH + "-01")
end_date = (pd.to_datetime(END_MONTH + "-31") + pd.offsets.MonthEnd(1))
print(start_date, end_date)


In [ ]:
# Step 2: Load TSV files and filter by date range

usecols = ["created_date", "body"]

comments_df = pd.read_csv(
    COMMENTS_FILE,
    sep="\t",
    usecols=usecols,
    parse_dates=["created_date"],
    dtype={"body": "string"},
    na_filter=False
)

submissions_df = pd.read_csv(
    SUBMISSIONS_FILE,
    sep="\t",
    usecols=usecols,
    parse_dates=["created_date"],
    dtype={"body": "string"},
    na_filter=False
)

# Filter by the specified date range
comments_df = comments_df[
    (comments_df["created_date"] >= start_date) &
    (comments_df["created_date"] <= end_date)
].copy()

submissions_df = submissions_df[
    (submissions_df["created_date"] >= start_date) &
    (submissions_df["created_date"] <= end_date)
].copy()

print("Comments rows after filtering:", len(comments_df))
print("Submissions rows after filtering:", len(submissions_df))


In [ ]:
# Step 3: Compute VADER compound scores for each body text

analyzer = SentimentIntensityAnalyzer()

def compute_vader_compound(text: str) -> float:
    """Return VADER compound sentiment score for a given text."""
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)
    scores = analyzer.polarity_scores(text)
    return scores["compound"]

# Apply sentiment scoring
comments_df["sentiment"] = comments_df["body"].apply(compute_vader_compound)
submissions_df["sentiment"] = submissions_df["body"].apply(compute_vader_compound)

comments_df.head(), submissions_df.head()


In [ ]:
# Step 4: Create year_month column and aggregate by month

# Create year_month in "YYYY-MM" format
comments_df["year_month"] = comments_df["created_date"].dt.to_period("M").astype(str)
submissions_df["year_month"] = submissions_df["created_date"].dt.to_period("M").astype(str)

# Aggregate comments: mean, sum, count
comments_monthly = (
    comments_df
    .groupby("year_month")["sentiment"]
    .agg(["mean", "sum", "count"])
    .rename(columns={
        "mean": "c_sentiment_score",
        "sum": "c_sentiment_sum",
        "count": "c_count"
    })
)

# Aggregate submissions: mean, sum, count
submissions_monthly = (
    submissions_df
    .groupby("year_month")["sentiment"]
    .agg(["mean", "sum", "count"])
    .rename(columns={
        "mean": "s_sentiment_score",
        "sum": "s_sentiment_sum",
        "count": "s_count"
    })
)

comments_monthly.head(), submissions_monthly.head()


In [ ]:
# Step 5: Merge monthly results and compute total sentiment score

# Outer join to keep months that appear in either comments or submissions
monthly = comments_monthly.join(submissions_monthly, how="outer")

# Replace NaN in sums and counts with 0
for col in ["c_sentiment_sum", "c_count", "s_sentiment_sum", "s_count"]:
    if col in monthly.columns:
        monthly[col] = monthly[col].fillna(0)

# Compute total sentiment score over all texts (comments + submissions)
monthly["total_sum"] = monthly["c_sentiment_sum"] + monthly["s_sentiment_sum"]
monthly["total_count"] = monthly["c_count"] + monthly["s_count"]

# Avoid division by zero
monthly["t_sentiment_score"] = monthly["total_sum"] / monthly["total_count"]
monthly.loc[monthly["total_count"] == 0, "t_sentiment_score"] = pd.NA

# Keep only requested columns and reset index
final_df = monthly.reset_index()[[
    "year_month",
    "c_sentiment_score",
    "s_sentiment_score",
    "t_sentiment_score"
]]

# Sort by year_month just in case
final_df = final_df.sort_values("year_month").reset_index(drop=True)

final_df.head()


In [ ]:
# Step 6: Write the result to a Google Spreadsheet

# Authenticate with Google account
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Open or create the spreadsheet
try:
    sh = gc.open(SPREADSHEET_NAME)
except gspread.SpreadsheetNotFound:
    sh = gc.create(SPREADSHEET_NAME)

# Open or create the worksheet
try:
    ws = sh.worksheet(WORKSHEET_NAME)
    ws.clear()
except gspread.WorksheetNotFound:
    # Create a new worksheet with enough rows/cols
    ws = sh.add_worksheet(title=WORKSHEET_NAME, rows="5000", cols="10")

# Write the DataFrame to the worksheet
set_with_dataframe(ws, final_df)

print(f"Written to spreadsheet: {SPREADSHEET_NAME}, worksheet: {WORKSHEET_NAME}")
